In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor
)
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)

# Local SQLite backend: runs, params, metrics and the model registry all live in mlflow.db
mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()

print("MLflow version:", mlflow.__version__)
print("Tracking URI  :", mlflow.get_tracking_uri())

MLflow version: 3.17.0
Tracking URI  : sqlite:///mlflow.db


In [2]:
df = pd.read_csv("real_estate_final.csv")

NUMERIC_COLS = [
    "BHK", "Size_in_SqFt", "Price_in_Lakhs", "Price_per_SqFt", "Year_Built",
    "Floor_No", "Total_Floors", "Age_of_Property", "Nearby_Schools",
    "Nearby_Hospitals", "Amenity_Density_Score", "Locality_Frequency"
]


def prepare(target, stratify=False):
    """Split + scale numeric columns (scaler fitted on train only)."""
    drop = ["Good_Investment", "Investment_Score", "Future_Price_5Y", "ID"]
    X = df.drop(columns=drop, errors="ignore")
    y = df[target]

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42,
        stratify=y if stratify else None
    )
    cols = [c for c in NUMERIC_COLS if c in X_tr.columns]
    X_tr, X_te = X_tr.copy(), X_te.copy()
    X_tr[cols] = X_tr[cols].astype(float)
    X_te[cols] = X_te[cols].astype(float)

    scaler = StandardScaler()
    X_tr[cols] = scaler.fit_transform(X_tr[cols])
    X_te[cols] = scaler.transform(X_te[cols])
    return X_tr, X_te, y_tr, y_te


Xc_train, Xc_test, yc_train, yc_test = prepare("Good_Investment", stratify=True)
Xr_train, Xr_test, yr_train, yr_test = prepare("Future_Price_5Y")

print("Classification:", Xc_train.shape, Xc_test.shape)
print("Regression    :", Xr_train.shape, Xr_test.shape)

Classification: (200000, 90) (50000, 90)
Regression    : (200000, 90) (50000, 90)


In [3]:
def clean_params(model):
    """Only simple, non-null hyper-parameters (MLflow params must be strings/numbers)."""
    out = {}
    for k, v in model.get_params().items():
        if v is None or isinstance(v, (str, int, float, bool)):
            out[k] = v
    return out


def classification_metrics(model, X, y):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, prob),
    }


def regression_metrics(model, X, y):
    pred = model.predict(X)
    mse = mean_squared_error(y, pred)
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": mean_absolute_error(y, pred),
        "r2": r2_score(y, pred),
    }


def log_one_run(name, model, task, X_train, X_test, y_train, y_test, metric_fn, extra_params=None):
    """Train one model inside its own MLflow run and log everything."""
    with mlflow.start_run(run_name=name) as run:
        model.fit(X_train, y_train)

        mlflow.set_tags({"task": task, "model_name": name})
        mlflow.log_params(clean_params(model))
        if extra_params:
            mlflow.log_params(extra_params)
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_train_rows", X_train.shape[0])

        scores = metric_fn(model, X_test, y_test)
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in scores.items()})

        signature = infer_signature(X_test.head(5), model.predict(X_test.head(5)))
        if name.startswith("XGBoost"):
            info = mlflow.xgboost.log_model(model, name="model", signature=signature, input_example=X_test.head(3))
        else:
            # cloudpickle: our own trusted models (MLflow 3 defaults to skops, which rejects tree models)
            info = mlflow.sklearn.log_model(
                model, name="model", signature=signature, input_example=X_test.head(3),
                serialization_format="cloudpickle",
            )
        mlflow.set_tag("model_uri", info.model_uri)   # used later for registration

        print(f"{name:<28} " + "  ".join(f"{k}={v:.4f}" for k, v in scores.items()))
        return run.info.run_id, scores

In [4]:
mlflow.set_experiment("Real_Estate_Classification")

neg, pos = (yc_train == 0).sum(), (yc_train == 1).sum()

clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=15, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, scale_pos_weight=neg / pos, eval_metric="logloss", random_state=42, n_jobs=-1),
}

for name, model in clf_models.items():
    log_one_run(name, model, "classification", Xc_train, Xc_test, yc_train, yc_test, classification_metrics)

2026/10/15 16:05:15 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/10/15 16:05:15 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

Logistic Regression          accuracy=0.9374  precision=0.9987  recall=0.9362  f1=0.9665  roc_auc=0.9873


2026/10/15 16:05:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree                accuracy=1.0000  precision=1.0000  recall=1.0000  f1=1.0000  roc_auc=0.9994


2026/10/15 16:06:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest                accuracy=0.9983  precision=0.9992  recall=0.9990  f1=0.9991  roc_auc=0.9997


2026/10/15 16:07:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Gradient Boosting            accuracy=0.9999  precision=0.9999  recall=1.0000  f1=1.0000  roc_auc=1.0000


XGBoost                      accuracy=1.0000  precision=1.0000  recall=1.0000  f1=1.0000  roc_auc=1.0000


In [5]:
# Tuning on a 40k sample keeps this fast; the final model is evaluated on the full test set
sample_idx = Xc_train.sample(n=min(40000, len(Xc_train)), random_state=42).index

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}

with mlflow.start_run(run_name="XGBoost Classifier Tuning") as parent:
    mlflow.set_tags({"task": "classification", "model_name": "XGBoost Tuned"})

    search = RandomizedSearchCV(
        XGBClassifier(scale_pos_weight=neg / pos, eval_metric="logloss", random_state=42, n_jobs=-1),
        param_dist, n_iter=8, cv=3, scoring="f1", random_state=42, n_jobs=1
    )
    search.fit(Xc_train.loc[sample_idx], yc_train.loc[sample_idx])

    # one nested child run per candidate
    for i, (params, score) in enumerate(zip(search.cv_results_["params"], search.cv_results_["mean_test_score"])):
        with mlflow.start_run(run_name=f"candidate_{i}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("cv_f1", float(score))

    mlflow.log_params({f"best_{k}": v for k, v in search.best_params_.items()})
    mlflow.log_metric("best_cv_f1", float(search.best_score_))
    print("Best params:", search.best_params_, "| CV F1:", round(search.best_score_, 4))

# Re-train the best configuration on the full train set as its own tracked run
tuned_clf = XGBClassifier(**search.best_params_, scale_pos_weight=neg / pos, eval_metric="logloss", random_state=42, n_jobs=-1)
log_one_run("XGBoost Tuned", tuned_clf, "classification", Xc_train, Xc_test, yc_train, yc_test, classification_metrics, extra_params={"tuned": True})

Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.2, 'colsample_bytree': 0.85} | CV F1: 0.9999


XGBoost Tuned                accuracy=1.0000  precision=1.0000  recall=1.0000  f1=1.0000  roc_auc=1.0000


('5a5b3babfb50431f9d4742170f211eb5',
 {'accuracy': 0.99998,
  'precision': 1.0,
  'recall': 0.9999792462228125,
  'f1': 0.9999896230037253,
  'roc_auc': 1.0})

In [6]:
mlflow.set_experiment("Real_Estate_Regression")

reg_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
}

for name, model in reg_models.items():
    log_one_run(name, model, "regression", Xr_train, Xr_test, yr_train, yr_test, regression_metrics)

2026/10/15 16:08:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression            mse=0.0000  rmse=0.0000  mae=0.0000  r2=1.0000


2026/10/15 16:08:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree                mse=0.0027  rmse=0.0517  mae=0.0445  r2=1.0000


2026/10/15 16:10:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest                mse=0.0000  rmse=0.0033  mae=0.0020  r2=1.0000


2026/10/15 16:12:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Gradient Boosting            mse=2.0195  rmse=1.4211  mae=1.0751  r2=1.0000


XGBoost                      mse=0.6630  rmse=0.8142  mae=0.7037  r2=1.0000


In [7]:
sample_idx = Xr_train.sample(n=min(40000, len(Xr_train)), random_state=42).index

with mlflow.start_run(run_name="XGBoost Regressor Tuning"):
    mlflow.set_tags({"task": "regression", "model_name": "XGBoost Tuned"})

    search_r = RandomizedSearchCV(
        XGBRegressor(random_state=42, n_jobs=-1),
        param_dist, n_iter=8, cv=3, scoring="r2", random_state=42, n_jobs=1
    )
    search_r.fit(Xr_train.loc[sample_idx], yr_train.loc[sample_idx])

    for i, (params, score) in enumerate(zip(search_r.cv_results_["params"], search_r.cv_results_["mean_test_score"])):
        with mlflow.start_run(run_name=f"candidate_{i}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("cv_r2", float(score))

    mlflow.log_params({f"best_{k}": v for k, v in search_r.best_params_.items()})
    mlflow.log_metric("best_cv_r2", float(search_r.best_score_))
    print("Best params:", search_r.best_params_, "| CV R2:", round(search_r.best_score_, 4))

tuned_reg = XGBRegressor(**search_r.best_params_, random_state=42, n_jobs=-1)
log_one_run("XGBoost Tuned", tuned_reg, "regression", Xr_train, Xr_test, yr_train, yr_test, regression_metrics, extra_params={"tuned": True})

Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 1.0} | CV R2: 1.0


XGBoost Tuned                mse=0.6653  rmse=0.8157  mae=0.7039  r2=1.0000


('37e4c27114974e72b82c24b406faad59',
 {'mse': 0.6653406113198153,
  'rmse': 0.8156841369793918,
  'mae': 0.7039169405867801,
  'r2': 0.9999845410773287})

In [8]:
def leaderboard(experiment, metric, ascending):
    runs = mlflow.search_runs(
        experiment_names=[experiment],
        filter_string="tags.model_name != '' and attributes.status = 'FINISHED'",
        order_by=[f"metrics.{metric} {'ASC' if ascending else 'DESC'}"],
    )
    runs = runs[runs[f"metrics.{metric}"].notna()]
    cols = ["run_id", "tags.model_name", "tags.model_uri"] + [c for c in runs.columns if c.startswith("metrics.test_")]
    return runs[cols].reset_index(drop=True)


clf_board = leaderboard("Real_Estate_Classification", "test_f1", ascending=False)
reg_board = leaderboard("Real_Estate_Regression", "test_r2", ascending=False)

print("Classification leaderboard (by F1)")
display(clf_board)
print("Regression leaderboard (by R2)")
display(reg_board)

Classification leaderboard (by F1)


,run_id,tags.model_name,tags.model_uri,metrics.test_roc_auc,metrics.test_precision,metrics.test_accuracy,metrics.test_f1,metrics.test_recall
0,5a5b3babfb50431f9d4742170f211eb5,XGBoost Tuned,models:/m-9fb77f3a55644fd09d976e143e25a13d,1.000000,1.000000,0.99998,0.999990,0.999979
1,cb57557163bc44d28f06c555a8f3c307,XGBoost,models:/m-04845091e03749ee983401ddf7759d8c,1.000000,1.000000,0.99998,0.999990,0.999979
2,7b801c133f14445e861f97b6ee46d10d,XGBoost Tuned,models:/m-34a79a10e807470a913de9b571f84872,1.000000,1.000000,0.99998,0.999990,0.999979
3,ca00dbbfa7ee4dd9b62a288812a5a26f,XGBoost,models:/m-af0f991df9954bd1b013b23c4bc28c79,1.000000,1.000000,0.99998,0.999990,0.999979
4,be5ea56f177948a195e117eca301e06c,XGBoost,models:/m-e46eaead297541e8a9c859983d806729,1.000000,1.000000,0.99998,0.999990,0.999979
5,eeadfb0f9eab402192f57751519fd629,Decision Tree,models:/m-385eac4c980347c984c170d6455970ec,0.999449,0.999958,0.99996,0.999979,1.000000
6,a188f540d87d45f9a93c0aef539090c0,Decision Tree,models:/m-99c0109a0edf4e0b8141ba1bc63d94b3,0.999449,0.999958,0.99996,0.999979,1.000000
7,e366eefc1d3d4a52bcea82a3547465d1,Decision Tree,models:/m-58d8a2ee05934faf93992ece3a7c57d8,0.999449,0.999958,0.99996,0.999979,1.000000
8,b3772ad952b24589a9ec12b481dc6a8b,Gradient Boosting,models:/m-a94ab1dd7994410699abd797a09dc608,1.000000,0.999938,0.99994,0.999969,1.000000
9,80b8befa693d4368a7a9de733e519041,Gradient Boosting,models:/m-2595d62f5247436081b398d1db0b60f0,1.000000,0.999938,0.99994,0.999969,1.000000


Regression leaderboard (by R2)


,run_id,tags.model_name,tags.model_uri,metrics.test_r2,metrics.test_mae,metrics.test_mse,metrics.test_rmse
0,a58d9fd227224eefbc712ce10c79b47d,Linear Regression,models:/m-61ee35a91f3c4138890681c5b0c183ff,1.000000,9.312014e-13,1.179289e-24,1.085951e-12
1,638cce8921a24851bbead640fe8da84a,Linear Regression,models:/m-6748f3e5c6524fc188fcdccae8d75066,1.000000,9.312014e-13,1.179289e-24,1.085951e-12
2,789863ff34184bd0a373c2c445b25138,Random Forest,models:/m-239aaf131570429a81e58576134459a8,1.000000,2.027801e-03,1.108327e-05,3.329154e-03
3,587832cb98614e3783c196d7cc7298a8,Random Forest,models:/m-18e3f92b0b0c4d59b01d955c5ebbbf83,1.000000,2.027801e-03,1.108327e-05,3.329154e-03
4,ebc19663fa824a27a7b9b923d7a4d8c8,Decision Tree,models:/m-d2eceec9b4be401383f12dae16807aa7,1.000000,4.449016e-02,2.670521e-03,5.167709e-02
5,e643d39240da4f17a20e0198f7f30a04,Decision Tree,models:/m-9e602f12654941e6b7e22fc8ddea3115,1.000000,4.449016e-02,2.670521e-03,5.167709e-02
6,fdb0ecf1176f4a3e9755ee04e97ae93b,XGBoost,models:/m-f9fa3bd4fdad42cb850f99980c666f56,0.999985,7.036928e-01,6.629573e-01,8.142219e-01
7,fb3754c5f55044adb824a7230cfcd080,XGBoost,models:/m-0c3ec0ec2bb948f4b9eda8d7d48eabf0,0.999985,7.036928e-01,6.629573e-01,8.142219e-01
8,37e4c27114974e72b82c24b406faad59,XGBoost Tuned,models:/m-27555747ce8c4c5a9a643a36b7195ecf,0.999985,7.039169e-01,6.653406e-01,8.156841e-01
9,a551a5cdeb554f5b9c9bae9f62e313db,XGBoost Tuned,models:/m-639f7924f80b4a298567beb8cdcb23fb,0.999985,7.039169e-01,6.653406e-01,8.156841e-01


In [9]:
def register_best(board, model_name):
    best_run_id = board.loc[0, "run_id"]
    best_label = board.loc[0, "tags.model_name"]

    result = mlflow.register_model(board.loc[0, "tags.model_uri"], model_name)
    client.set_registered_model_alias(model_name, "champion", result.version)

    print(f"{model_name}: registered '{best_label}' as version {result.version} -> alias @champion")
    return result.version


clf_version = register_best(clf_board, "RealEstateInvestmentClassifier")
reg_version = register_best(reg_board, "RealEstatePriceRegressor")

RealEstateInvestmentClassifier: registered 'XGBoost Tuned' as version 2 -> alias @champion
RealEstatePriceRegressor: registered 'Linear Regression' as version 2 -> alias @champion


Registered model 'RealEstateInvestmentClassifier' already exists. Creating a new version of this model...
Created version '2' of model 'RealEstateInvestmentClassifier'.
Registered model 'RealEstatePriceRegressor' already exists. Creating a new version of this model...
Created version '2' of model 'RealEstatePriceRegressor'.


In [10]:
clf_champion = mlflow.pyfunc.load_model("models:/RealEstateInvestmentClassifier@champion")
reg_champion = mlflow.pyfunc.load_model("models:/RealEstatePriceRegressor@champion")

sample_c = Xc_test.head(5)
sample_r = Xr_test.head(5)

print("Investment class  :", clf_champion.predict(sample_c))
print("Actual class      :", yc_test.head(5).tolist())
print()
print("Future price (L)  :", np.round(reg_champion.predict(sample_r), 2))
print("Actual price (L)  :", np.round(yr_test.head(5).values, 2))

Investment class  : [1 1 1 1 1]
Actual class      : [1, 1, 1, 1, 1]

Future price (L)  : [377.06 126.42 349.49 498.7  183.65]
Actual price (L)  : [377.06 126.42 349.49 498.7  183.65]
